In [ ]:
# STEP 1: CHECK WHAT FILES ARE AVAILABLE IN CURRENT DIRECTORY

import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir())

Current working directory:
/content

Files in current directory:
['.config', 'conll2003', 'archive (4).zip', 'results', 'sample_data']


In [ ]:
# STEP 2: EXTRACT CoNLL-2003 DATASET FROM ZIP FILE

import zipfile
import os

zip_path = "/content/archive (4).zip"
extract_path = "/content/conll2003"

# Create folder if not exists
os.makedirs(extract_path, exist_ok=True)

# Extract files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Check extracted files
print("Extracted files:")
print(os.listdir(extract_path))

Extracted files:
['conll2003']


In [ ]:
# STEP 3: OPEN INNER FOLDER AND LIST ALL FILES

import os

inner_path = "/content/conll2003/conll2003"

files = os.listdir(inner_path)
print("Files inside inner folder:")
print(files)

Files inside inner folder:
['eng.testa', 'eng.train', 'eng.testb']


In [ ]:
# STEP 4: READ CoNLL-2003 FILE AND CONVERT INTO TOKENS + LABELS FORMAT

def read_conll_file(file_path):
    sentences = []
    labels = []

    tokens = []
    ner_tags = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # If empty line → sentence ends
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    tokens = []
                    ner_tags = []
            else:
                parts = line.split()

                # Format: word POS CHUNK NER
                token = parts[0]
                ner = parts[-1]   # last column = NER (chunk-like)

                tokens.append(token)
                ner_tags.append(ner)

    return sentences, labels


# Paths
train_path = "/content/conll2003/conll2003/eng.train"
val_path = "/content/conll2003/conll2003/eng.testa"
test_path = "/content/conll2003/conll2003/eng.testb"

# Load data
train_sentences, train_labels = read_conll_file(train_path)
val_sentences, val_labels = read_conll_file(val_path)
test_sentences, test_labels = read_conll_file(test_path)

# Check sample
print("Sample sentence:", train_sentences[0])
print("Sample labels:", train_labels[0])

Sample sentence: ['-DOCSTART-']
Sample labels: ['O']


In [ ]:
# STEP 5: REMOVE '-DOCSTART-' SENTENCES FROM DATA

def clean_data(sentences, labels):
    clean_sentences = []
    clean_labels = []

    for sent, lab in zip(sentences, labels):
        if sent[0] != "-DOCSTART-":
            clean_sentences.append(sent)
            clean_labels.append(lab)

    return clean_sentences, clean_labels


# Apply cleaning
train_sentences, train_labels = clean_data(train_sentences, train_labels)
val_sentences, val_labels = clean_data(val_sentences, val_labels)
test_sentences, test_labels = clean_data(test_sentences, test_labels)

# Check again
print("Cleaned sample sentence:", train_sentences[0])
print("Cleaned sample labels:", train_labels[0])

Cleaned sample sentence: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Cleaned sample labels: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [ ]:
# STEP 6: CREATE LABEL MAPPING (IMPORTANT FOR MODEL)

# Get all unique labels
unique_labels = set()

for label_list in train_labels:
    unique_labels.update(label_list)

# Convert to sorted list
label_list = sorted(list(unique_labels))

# Create mappings
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Print results
print("Label List:", label_list)
print("Label2ID:", label2id)

Label List: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Label2ID: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [ ]:
# STEP 7: CONVERT DATA INTO HUGGING FACE DATASET FORMAT

from datasets import Dataset, DatasetDict

# Convert labels to IDs
def convert_labels_to_ids(labels):
    return [[label2id[label] for label in sentence] for sentence in labels]

train_labels_ids = convert_labels_to_ids(train_labels)
val_labels_ids = convert_labels_to_ids(val_labels)
test_labels_ids = convert_labels_to_ids(test_labels)

# Create datasets
train_dataset = Dataset.from_dict({
    "tokens": train_sentences,
    "labels": train_labels_ids
})

val_dataset = Dataset.from_dict({
    "tokens": val_sentences,
    "labels": val_labels_ids
})

test_dataset = Dataset.from_dict({
    "tokens": test_sentences,
    "labels": test_labels_ids
})

# Combine into DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# Check sample
print(dataset)
print("\nSample:", dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 3453
    })
})

Sample: {'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'labels': [2, 8, 1, 8, 8, 8, 1, 8, 8]}


In [ ]:
# STEP 8: TOKENIZATION + LABEL ALIGNMENT

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # Special tokens
        elif word_idx != previous_word_idx:
            labels.append(example["labels"][word_idx])  # First token of word
        else:
            labels.append(-100)  # Subword tokens

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


# Apply tokenization
tokenized_dataset = dataset.map(tokenize_and_align_labels)

# Check sample
print(tokenized_dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'labels': [-100, 2, 8, 1, 8, 8, 8, 1, 8, 8, -100], 'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
# STEP 9: LOAD BERT MODEL FOR TOKEN CLASSIFICATION

from transformers import AutoModelForTokenClassification

num_labels = len(label_list)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Model loaded successfully!


In [ ]:
# STEP 10:  TRAINING SETTINGS

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,      # only 1 epoch
    logging_steps=200
)

In [ ]:
# STEP 11A: INSTALL REQUIRED LIBRARY (SEQEVAL METRICS)

!pip install evaluate seqeval

In [ ]:
# STEP 11 FIX: RECREATE LABEL MAPPINGS FROM DATA

# Collect unique labels again from training data
unique_labels = set()

for example in tokenized_dataset["train"]:
    for label in example["labels"]:
        if label != -100:
            unique_labels.add(label)

# Convert to sorted list
label_list = sorted(list(unique_labels))

# Create mappings
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print("Recreated label list:", label_list)

Recreated label list: [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [ ]:
# STEP 11: LOAD DISTILBERT (FAST MODEL)

from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# STEP 12: USE SMALL SUBSET FOR FAST TRAINING

small_train = tokenized_dataset["train"].select(range(2000))
small_val = tokenized_dataset["validation"].select(range(500))

print("Train size:", len(small_train))
print("Validation size:", len(small_val))

Train size: 2000
Validation size: 500


In [ ]:
# STEP 13 FIX: DEFINE METRICS FUNCTION AGAIN

import numpy as np
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                current_preds.append(id2label[p_val])
                current_labels.append(id2label[l_val])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

In [ ]:
# STEP 13: FINAL TRAINER SETUP (FAST)

from transformers import Trainer, DataCollatorForTokenClassification

# Data collator for padding
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
200,0.339102


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.3020228462219238, metrics={'train_runtime': 966.5027, 'train_samples_per_second': 2.069, 'train_steps_per_second': 0.259, 'total_flos': 20674386567072.0, 'train_loss': 0.3020228462219238, 'epoch': 1.0})

In [ ]:
# FINAL FIX: FORCE CONVERT IDS → LABEL STRINGS

import torch
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

# 🔥 Explicit mapping (VERY IMPORTANT)
id2label = {
    0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER',
    4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'
}

model.eval()

true_predictions = []
true_labels = []

for example in small_val:
    inputs = {
        "input_ids": torch.tensor([example["input_ids"]]),
        "attention_mask": torch.tensor([example["attention_mask"]])
    }

    with torch.no_grad():
        outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2).numpy()[0]
    labels = example["labels"]

    pred_labels = []
    true_lab = []

    for p, l in zip(preds, labels):
        if l != -100:
            pred_labels.append(id2label[int(p)])   # 🔥 force string
            true_lab.append(id2label[int(l)])      # 🔥 force string

    true_predictions.append(pred_labels)
    true_labels.append(true_lab)

# Compute metrics
results = metric.compute(predictions=true_predictions, references=true_labels)

print("Precision:", results["overall_precision"])
print("Recall:", results["overall_recall"])
print("F1 Score:", results["overall_f1"])

Precision: 0.7759914255091104
Recall: 0.8227272727272728
F1 Score: 0.7986762272476557


In [ ]:
# STEP 16: CUSTOM SENTENCE PREDICTION

sentence = "John works at Google in California"

tokens = sentence.split()

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

import torch
with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2).numpy()[0]

# Mapping
id2label = {
    0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER',
    4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'
}

print("Predictions:\n")
for token, pred in zip(tokens, predictions[1:len(tokens)+1]):
    print(f"{token} → {id2label[int(pred)]}")

Predictions:

John → B-PER
works → O
at → O
Google → B-ORG
in → O
California → B-LOC


Comparison

| Feature    | POS Tagging              | Chunking                            |
| ---------- | ------------------------ | ----------------------------------- |
| Level      | Word-level               | Phrase-level                        |
| Purpose    | Assign grammatical roles | Group words into meaningful phrases |
| Example    | Noun, Verb, Adjective    | Noun Phrase (NP), Verb Phrase (VP)  |
| Complexity | Easy                     | Medium                              |
| Output     | Each word gets a tag     | Group of words gets a chunk label   |
